%md%md

# 🥈 Tratamento Silver - Ecommerce Pedidos

## Squad 2 - Real Time for Business

### Grupo 3 - Ecommerce Pedidos

---

# 🎯 Objetivo

Este notebook tem como objetivo realizar o tratamento,
padronização e validação dos dados provenientes da camada
Bronze.

Nesta etapa, os dados ingeridos anteriormente serão
transformados em uma camada confiável e preparada para
consumo analítico.

---

# 🏗️ Contexto da Arquitetura

O projeto utiliza arquitetura baseada em camadas:

```
Raw

↓

🥉 Bronze

↓

🥈 Silver

↓

🥇 Gold
```

Cada camada possui uma responsabilidade específica no
pipeline de dados.

---

# 📥 Origem dos Dados

Os dados utilizados nesta etapa são provenientes da camada
Bronze.

Origem:

```
Database:

grupo3_bronze


Tabela:

ecommerce_pedidos
```

A Bronze contém:

- Dados originais;
- Informações de auditoria;
- Rastreamento da origem dos arquivos.

---

# 🥈 Camada Silver

A camada Silver é responsável por transformar os dados
brutos em dados tratados e padronizados.

Nesta etapa serão realizados:

- Correção de inconsistências;
- Padronização de valores;
- Validação dos dados;
- Preparação para análises futuras.

---

# 🧹 Tratamentos Aplicados

## 💳 Padronização de Pagamentos

Durante a análise da Bronze foram identificados diferentes
formatos de pagamento:

Exemplo:

```
pix

boleto

cartão débito

cartão crédito
```

Na Silver esses valores serão padronizados:

```
Pix

Boleto

Cartão
```

---

# 📅 Tratamento de Datas

As informações temporais serão validadas:

```
dt_pedido

dt_previsao_entrega

dt_ultima_atualizacao_status
```

Objetivos:

- Garantir tipos corretos;
- Permitir análises temporais;
- Melhorar consultas futuras.

---

# 🔍 Validação da Qualidade

Serão realizadas validações:

- Registros nulos;
- Duplicidades;
- Consistência dos dados;
- Valores inválidos;
- Integridade das informações.

---

# 📊 Estrutura dos Dados Silver

Após os tratamentos, os dados estarão preparados para:

- Análises;
- Relatórios;
- Indicadores;
- Camada Gold.

---

# 📍 Destino dos Dados

Os dados tratados serão armazenados como Delta Lake.

Destino:

```
Database:

grupo3_silver


Tabela:

ecommerce_pedidos
```

---

# 🔄 Fluxo de Transformação

```
grupo3_bronze.ecommerce_pedidos

        ↓

Tratamentos

        ↓

Validações

        ↓

grupo3_silver.ecommerce_pedidos
```

---

# 🛡️ Boas Práticas Aplicadas

Nesta etapa são aplicadas práticas de Engenharia de Dados:

- Separação entre ingestão e transformação;
- Padronização dos dados;
- Controle de qualidade;
- Organização por camadas;
- Preparação para consumo analítico.

---

# ✅ Resultado Esperado

Ao final deste notebook:

- Os dados Bronze estarão tratados;
- Os pagamentos estarão padronizados;
- As informações estarão validadas;
- A tabela Silver estará disponível para análises.

Este notebook representa a etapa de tratamento dos dados
da Squad 2 - Grupo 3.


In [0]:
# ==================================================
# IMPORTAÇÃO DAS BIBLIOTECAS
# TRATAMENTO SILVER
# ==================================================


# Biblioteca para manipulação
# de dados em formato tabular.

import pandas as pd



# Biblioteca padrão Python
# para acesso às variáveis de ambiente.

import os



# Biblioteca para carregamento
# do arquivo .env.

from dotenv import load_dotenv



# Funções Spark utilizadas
# nos tratamentos da Silver.

from pyspark.sql.functions import (

    col,

    when,

    count,

    lit,

    current_timestamp

)



print(
    "Bibliotecas Silver carregadas com sucesso!"
)

Bibliotecas Silver carregadas com sucesso!


In [0]:
# ==================================================
# CONFIGURAÇÃO DO AMBIENTE
# TRATAMENTO SILVER
# ==================================================


# Carrega as variáveis do arquivo .env.

load_dotenv(
    ".env",
    override=True
)



# Define a origem dos dados.

bronze_database = (
    "grupo3_bronze"
)


bronze_table = (
    "ecommerce_pedidos"
)



# Define o destino Silver.

silver_database = (
    "grupo3_silver"
)


silver_table = (
    "ecommerce_pedidos"
)



print(
    "Configurações Silver carregadas com sucesso!"
)



print(
    "Origem Bronze:",
    f"{bronze_database}.{bronze_table}"
)



print(
    "Destino Silver:",
    f"{silver_database}.{silver_table}"
)

Configurações Silver carregadas com sucesso!
Origem Bronze: grupo3_bronze.ecommerce_pedidos
Destino Silver: grupo3_silver.ecommerce_pedidos


In [0]:
# ==================================================
# LEITURA DA CAMADA BRONZE
# TRATAMENTO SILVER
# ==================================================


# Carrega a tabela Delta Bronze
# criada no notebook de ingestão.

df_silver = (
    spark
    .table(
        f"{bronze_database}.{bronze_table}"
    )
)



print(
    "Tabela Bronze carregada com sucesso!"
)



print(
    "Quantidade de registros:",
    df_silver.count()
)



print(
    "Quantidade de colunas:",
    len(df_silver.columns)
)

Tabela Bronze carregada com sucesso!
Quantidade de registros: 27
Quantidade de colunas: 12


In [0]:
# ==================================================
# ANÁLISE DO SCHEMA BRONZE
# ANTES DOS TRATAMENTOS
# ==================================================


print(
    "Schema da camada Bronze:"
)



df_silver.printSchema()

Schema da camada Bronze:
root
 |-- id_pedido: long (nullable = true)
 |-- id_cliente: long (nullable = true)
 |-- id_endereco_entrega: long (nullable = true)
 |-- dt_pedido: timestamp (nullable = true)
 |-- status_pedido: string (nullable = true)
 |-- valor_total: double (nullable = true)
 |-- valor_frete: double (nullable = true)
 |-- metodo_pagamento: string (nullable = true)
 |-- dt_previsao_entrega: timestamp (nullable = true)
 |-- dt_ultima_atualizacao_status: timestamp (nullable = true)
 |-- bronze_source_file: string (nullable = true)
 |-- bronze_ingested_at: timestamp (nullable = true)



In [0]:
# ==================================================
# PADRONIZAÇÃO DOS MÉTODOS DE PAGAMENTO
# CAMADA SILVER
# ==================================================


# Cria uma nova regra de padronização
# para o campo metodo_pagamento.
#
# O objetivo é reduzir variações
# e facilitar análises futuras.


df_silver = (
    df_silver
    .withColumn(

        "metodo_pagamento",

        when(
            col("metodo_pagamento")
            .isin(
                "cartão débito",
                "cartão crédito"
            ),
            "Cartão"
        )

        .when(
            col("metodo_pagamento")
            == "pix",
            "Pix"
        )

        .when(
            col("metodo_pagamento")
            == "boleto",
            "Boleto"
        )

        .otherwise(
            col("metodo_pagamento")
        )
    )
)



print(
    "Métodos de pagamento padronizados com sucesso!"
)

Métodos de pagamento padronizados com sucesso!


In [0]:
# ==================================================
# VALIDAÇÃO DOS MÉTODOS DE PAGAMENTO
# CAMADA SILVER
# ==================================================


# Agrupa os pagamentos após
# a padronização realizada.

pagamentos_silver = (
    df_silver
    .groupBy(
        "metodo_pagamento"
    )
    .count()
    .orderBy(
        col("count").desc()
    )
)



display(
      pagamentos_silver
)

metodo_pagamento,count
Boleto,10
Cartão,10
Pix,7


In [0]:
# ==================================================
# ADIÇÃO DO METADADO SILVER
# ==================================================


# Adiciona o momento em que o tratamento Silver
# foi executado.

df_silver = (
    df_silver
    .withColumn(
        "silver_processed_at",
        current_timestamp()
    )
)



print(
    "Metadado silver_processed_at adicionado com sucesso!"
)



print(
    "Quantidade de colunas:",
    len(df_silver.columns)
)

Metadado silver_processed_at adicionado com sucesso!
Quantidade de colunas: 13


In [0]:
# ==================================================
# VALIDAÇÃO DE QUALIDADE SILVER
# ==================================================


# Quantidade total de registros.

total_registros_silver = (
    df_silver
    .count()
)



# Quantidade de pedidos únicos.

pedidos_unicos_silver = (
    df_silver
    .select(
        "id_pedido"
    )
    .distinct()
    .count()
)



# Calcula duplicidades.

duplicidades_silver = (
    total_registros_silver
    -
    pedidos_unicos_silver
)



print(
    "Total de registros:",
    total_registros_silver
)


print(
    "Pedidos únicos:",
    pedidos_unicos_silver
)


print(
    "Duplicidades:",
    duplicidades_silver
)

Total de registros: 27
Pedidos únicos: 27
Duplicidades: 0


In [0]:
# ==================================================
# VALIDAÇÃO DE VALORES NULOS
# CAMADA SILVER
# ==================================================


# Campos críticos para validação.

campos_validacao = [

    "id_pedido",

    "valor_total",

    "metodo_pagamento",

    "status_pedido"

]



print(
    "Quantidade de valores nulos:"
)



for campo in campos_validacao:


    nulos = (
        df_silver
        .filter(
            col(campo)
            .isNull()
        )
        .count()
    )


    print(
        campo,
        ":",
        nulos
    )

Quantidade de valores nulos:
id_pedido : 0
valor_total : 0
metodo_pagamento : 0
status_pedido : 0


In [0]:
# ==================================================
# SCHEMA FINAL SILVER
# ==================================================


print(
    "Schema final da camada Silver:"
)



df_silver.printSchema()

Schema final da camada Silver:
root
 |-- id_pedido: long (nullable = true)
 |-- id_cliente: long (nullable = true)
 |-- id_endereco_entrega: long (nullable = true)
 |-- dt_pedido: timestamp (nullable = true)
 |-- status_pedido: string (nullable = true)
 |-- valor_total: double (nullable = true)
 |-- valor_frete: double (nullable = true)
 |-- metodo_pagamento: string (nullable = true)
 |-- dt_previsao_entrega: timestamp (nullable = true)
 |-- dt_ultima_atualizacao_status: timestamp (nullable = true)
 |-- bronze_source_file: string (nullable = true)
 |-- bronze_ingested_at: timestamp (nullable = true)
 |-- silver_processed_at: timestamp (nullable = false)



In [0]:
# ==================================================
# CRIAÇÃO DO DATABASE SILVER
# ==================================================


spark.sql(
    """
    CREATE DATABASE IF NOT EXISTS grupo3_silver
    """
)



print(
    "Database Silver criado/verificado com sucesso!"
)

Database Silver criado/verificado com sucesso!


In [0]:
# ==================================================
# GRAVAÇÃO DA CAMADA SILVER
# FORMATO DELTA LAKE
# ==================================================


# Salva os dados tratados como uma tabela Delta.

(
    df_silver
    .write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(
        "grupo3_silver.ecommerce_pedidos"
    )
)



print(
    "Tabela Silver Delta criada com sucesso!"
)

Tabela Silver Delta criada com sucesso!


In [0]:
# ==================================================
# VALIDAÇÃO FINAL DA TABELA SILVER
# ==================================================


# Carrega a tabela Silver criada.

df_validacao_silver = (
    spark
    .table(
        "grupo3_silver.ecommerce_pedidos"
    )
)



print(
    "Tabela Silver carregada para validação!"
)



print(
    "Quantidade de registros:",
    df_validacao_silver.count()
)



print(
    "Quantidade de colunas:",
    len(df_validacao_silver.columns)
)

Tabela Silver carregada para validação!
Quantidade de registros: 27
Quantidade de colunas: 13


In [0]:
# ==================================================
# VISUALIZAÇÃO DOS DADOS SILVER TRATADOS
# ==================================================


display(
    df_validacao_silver
    .select(

        "id_pedido",

        "valor_total",

        "metodo_pagamento",

        "bronze_source_file",

        "bronze_ingested_at",

        "silver_processed_at"

    )
    .limit(10)
)

id_pedido,valor_total,metodo_pagamento,bronze_source_file,bronze_ingested_at,silver_processed_at
1790000559,65.0,Boleto,real-time-data/2026/09/21/112245/ecommerce_pedidos.parquet,2026-09-29T14:20:03.906Z,2026-09-29T14:22:32.867Z
1790000560,128.12,Boleto,real-time-data/2026/09/21/112245/ecommerce_pedidos.parquet,2026-09-29T14:20:03.906Z,2026-09-29T14:22:32.867Z
1790000561,271.93,Cartão,real-time-data/2026/09/21/112245/ecommerce_pedidos.parquet,2026-09-29T14:20:03.906Z,2026-09-29T14:22:32.867Z
1790000562,18.66,Boleto,real-time-data/2026/09/21/112245/ecommerce_pedidos.parquet,2026-09-29T14:20:03.906Z,2026-09-29T14:22:32.867Z
1790000563,221.31,Pix,real-time-data/2026/09/21/112245/ecommerce_pedidos.parquet,2026-09-29T14:20:03.906Z,2026-09-29T14:22:32.867Z
1790000564,49.64,Pix,real-time-data/2026/09/21/112245/ecommerce_pedidos.parquet,2026-09-29T14:20:03.906Z,2026-09-29T14:22:32.867Z
1790000640,664.13,Pix,real-time-data/2026/09/21/112402/ecommerce_pedidos.parquet,2026-09-29T14:20:03.906Z,2026-09-29T14:22:32.867Z
1790000641,68.38,Cartão,real-time-data/2026/09/21/112402/ecommerce_pedidos.parquet,2026-09-29T14:20:03.906Z,2026-09-29T14:22:32.867Z
1790000760,237.38,Cartão,real-time-data/2026/09/21/112602/ecommerce_pedidos.parquet,2026-09-29T14:20:03.906Z,2026-09-29T14:22:32.867Z
1790000761,101.15,Cartão,real-time-data/2026/09/21/112602/ecommerce_pedidos.parquet,2026-09-29T14:20:03.906Z,2026-09-29T14:22:32.867Z


%md

# ✅ Conclusão - Tratamento Silver de Pedidos

## Squad 2 - Real Time for Business

### Grupo 3 - Ecommerce Pedidos

---

# 🎯 Objetivo Alcançado

Este notebook realizou o tratamento e a padronização dos
dados provenientes da camada Bronze.

Os dados foram transformados em uma camada confiável,
organizada e preparada para consumo analítico.

---

# 📥 Origem dos Dados

Os dados utilizados foram carregados da camada Bronze:

```
Database:

grupo3_bronze


Tabela:

ecommerce_pedidos
```

A camada Bronze continha:

- Dados ingeridos do Real Time;
- Informações de auditoria;
- Identificação dos arquivos de origem.

---

# 🥉 Bronze → 🥈 Silver

Fluxo realizado:

```
grupo3_bronze.ecommerce_pedidos

        ↓

Tratamentos e validações

        ↓

grupo3_silver.ecommerce_pedidos
```

---

# 🧹 Tratamentos Realizados

## 💳 Padronização dos Pagamentos

Durante a análise da Bronze foram identificadas diferentes
representações para métodos de pagamento.

Exemplo:

Antes:

```
pix

boleto

cartão débito

cartão crédito
```

Depois:

```
Pix

Boleto

Cartão
```

Essa padronização melhora a qualidade dos dados e facilita
análises futuras.

---

# ⏰ Controle de Processamento

Foi adicionado o campo:

```
silver_processed_at
```

Esse campo registra o momento em que os dados foram tratados
pela camada Silver.

---

# 🏷️ Rastreabilidade Mantida

A Silver manteve os controles de auditoria da Bronze:

```
bronze_source_file

bronze_ingested_at
```

Com isso é possível identificar:

- Qual arquivo originou o registro;
- Quando o dado entrou na Bronze;
- Quando o tratamento Silver foi realizado.

---

# 🔍 Validação da Qualidade

Foram realizadas validações:

## Registros

```
27 registros processados
```

---

## Duplicidades

Resultado:

```
0 duplicidades encontradas
```

---

## Valores Nulos

Campos críticos validados:

```
id_pedido

valor_total

metodo_pagamento

status_pedido
```

Resultado:

```
0 valores nulos encontrados
```

---

# 📊 Estrutura Final Silver

Tabela criada:

```
Database:

grupo3_silver


Tabela:

ecommerce_pedidos
```

Quantidade final:

```
Registros:

27


Colunas:

13
```

---

# 🥈 Características da Camada Silver

A camada Silver agora possui:

- Dados tratados;
- Valores padronizados;
- Tipos validados;
- Controle de auditoria;
- Estrutura preparada para análise.

---

# 🛡️ Boas Práticas Aplicadas

Durante esta etapa foram aplicadas práticas de Engenharia
de Dados:

- Separação entre ingestão e tratamento;
- Padronização dos dados;
- Validação de qualidade;
- Controle de rastreabilidade;
- Organização por camadas;
- Persistência em Delta Lake.

---

# 🔄 Próxima Etapa do Pipeline

Após a conclusão da Silver, os dados seguirão para a camada:

```
🥇 Gold
```

Responsável por criar:

- Indicadores de negócio;
- Métricas analíticas;
- Tabelas preparadas para consumo.

---

# 🚀 Resultado Final

A camada Silver foi concluída com sucesso.

Os dados provenientes da Bronze foram tratados,
padronizados e validados, ficando preparados para
a próxima etapa do pipeline de Engenharia de Dados
da Squad 2 - Grupo 3.